# Laboratorio #9 - Embeddings

**Juan Diego Letona Aguilar**

*20230285*

In [39]:
# Notebook 9 (el pdf dice laboratorio 8, pero corresponde al laboratorio 9).
# la primera parte (imports, carga de paquetes, etc.) es parecida a las iteraciones anteriores
# principalmente librerias y carga de paquetes necesarios para el proyecto

import sys
import subprocess

packages = [
    "pandas",
    "numpy",
    "nltk",
    "matplotlib",
    "scikit-learn",
    "scipy",
    "gensim",
    "torch",
    "tqdm"
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", *packages
])

0

In [40]:
from pathlib import Path
from collections import Counter
import re
import time
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.tokenize import sent_tokenize
from tqdm.auto import tqdm

from gensim.models import KeyedVectors, Word2Vec, FastText

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)

#semilla para que los resultados sean reproducibles
SEED = 42
np.random.seed(SEED)

#recursos de NLTK
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


## 1. Preparación de los corpus

**Carga de los corpus**

In [41]:
#cargamos el corpus de noticias (mismo dataset y misma limpieza basica de los laboratorios 1 y 2)

df = pd.read_csv("df_total.csv")

URL_COL = "url"
TEXT_COL = "news"
CATEGORY_COL = "Type"

corpus = df[[URL_COL, TEXT_COL, CATEGORY_COL]].copy()

corpus[TEXT_COL] = corpus[TEXT_COL].fillna("").astype(str)
corpus = corpus[corpus[TEXT_COL].str.strip().ne("")]
corpus = corpus.drop_duplicates(subset=[TEXT_COL]).reset_index(drop=True)

print("Dimensiones del DataFrame original:", df.shape)
print("Documentos de noticias despues de quitar vacios y duplicados:", len(corpus))

Dimensiones del DataFrame original: (1217, 3)
Documentos de noticias despues de quitar vacios y duplicados: 1137


In [42]:
#cargamos el Quijote (texto de Project Gutenberg)

with open("don-quijote.txt", encoding="utf-8-sig") as archivo:
    texto_quijote = archivo.read()

#quitamos la licencia y el encabezado de gutenberg, que no forman parte de la obra
inicio = texto_quijote.find("*** START")
fin = texto_quijote.find("*** END")
texto_quijote = texto_quijote[inicio:fin]
texto_quijote = texto_quijote.split("\n", 1)[1]

#cada capitulo es un documento
documentos_quijote = re.split(r"(?m)^\s*Cap[ií]tulo\s+", texto_quijote)
documentos_quijote = [doc for doc in documentos_quijote if doc.strip()]

print("Caracteres del Quijote:", f"{len(texto_quijote):,}")
print("Documentos del Quijote (capitulos y preliminares):", len(documentos_quijote))
print(documentos_quijote[1][:300])

Caracteres del Quijote: 2,098,169
Documentos del Quijote (capitulos y preliminares): 127
primero. Que trata de la condición y ejercicio del famoso hidalgo
don Quijote de la Mancha


En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha mucho
tiempo que vivía un hidalgo de los de lanza en astillero, adarga antigua,
rocín flaco y galgo corredor. Una olla de algo más vaca que


**Segmentación en oraciones y tokenización**

Primero se arreglan algunos detalles del texto crudo. En las noticias hay lugares donde falta el espacio después del punto (por ejemplo "imaginamos.Para"), lo cual confunde al segmentador. En el Quijote el texto viene cortado en renglones, así que se unen antes de segmentar.

In [43]:
#funciones para segmentar en oraciones y tokenizar en palabras

patron_palabras = re.compile(r"[a-záéíóúüñ]+")

def segmentar_oraciones(texto):
    #agregamos el espacio que falta despues de un punto pegado a una mayuscula
    texto = re.sub(r"([a-záéíóúñ])([.!?])([A-ZÁÉÍÓÚÑ¿¡])", r"\1\2 \3", texto)
    #unimos los renglones y los espacios multiples
    texto = re.sub(r"\s+", " ", texto).strip()
    return sent_tokenize(texto, language="spanish")

def tokenizar(oracion):
    #minusculas y solo palabras (letras con tilde, ñ y ü), sin puntuacion ni numeros
    return patron_palabras.findall(oracion.lower())

def preparar_corpus(documentos):
    #devuelve una lista de listas de tokens (una lista por oracion) y el numero de oraciones por documento
    oraciones_tokens = []
    oraciones_por_doc = []

    for documento in documentos:
        tokens_doc = [tokenizar(oracion) for oracion in segmentar_oraciones(documento)]
        #descartamos oraciones con menos de 2 tokens porque no tienen contexto
        tokens_doc = [oracion for oracion in tokens_doc if len(oracion) >= 2]
        oraciones_tokens.extend(tokens_doc)
        oraciones_por_doc.append(len(tokens_doc))

    return oraciones_tokens, oraciones_por_doc

oraciones_noticias, oraciones_por_doc_noticias = preparar_corpus(
    tqdm(corpus[TEXT_COL], desc="Noticias")
)
oraciones_quijote, oraciones_por_doc_quijote = preparar_corpus(
    tqdm(documentos_quijote, desc="Quijote")
)

print("Ejemplo de oracion de noticias:")
print(oraciones_noticias[10])
print("\nEjemplo de oracion del Quijote:")
print(oraciones_quijote[10])

Quijote: 100%|██████████| 127/127 [00:00<00:00, 483.62it/s]

Ejemplo de oracion de noticias:
['la', 'comisión', 'reguladora', 'de', 'valores', 'de', 'china', 'csrc', 'dijo', 'en', 'un', 'comunicado', 'que', 'tomó', 'nota', 'de', 'los', 'nuevos', 'requisitos', 'de', 'la', 'comisión', 'de', 'valores', 'de', 'estados', 'unidos', 'sec', 'para', 'las', 'empresas', 'chinas', 'y', 'que', 'ambas', 'partes', 'deberían', 'mantener', 'el', 'espíritu', 'de', 'respeto', 'mutuo', 'y', 'reforzar', 'las', 'comunicaciones', 'sobre', 'la', 'regulación']

Ejemplo de oracion del Quijote:
['la', 'cual', 'dicha', 'pena', 'sea', 'la', 'tercia', 'parte', 'para', 'la', 'persona', 'que', 'lo', 'acusare', 'y', 'la', 'otra', 'tercia', 'parte', 'para', 'nuestra', 'cámara', 'y', 'la', 'otra', 'tercia', 'parte', 'para', 'el', 'juez', 'que', 'lo', 'sentenciare']


**Normalización: decisiones y compatibilidad con fastText SUC**

In [44]:
#cargamos los vectores preentrenados de fastText SUC (tamaño M, 100 dimensiones)
#solo las 200,000 palabras mas frecuentes para no llenar la memoria

RUTA_SUC = Path("embeddings-m-model.vec")
URL_SUC = "https://zenodo.org/record/3234051/files/embeddings-m-model.vec?download=1"

if not RUTA_SUC.exists():
    print("Descargando embeddings-m-model.vec (1.1 GB), esto tarda un rato...")
    urllib.request.urlretrieve(URL_SUC, RUTA_SUC)

suc = KeyedVectors.load_word2vec_format(str(RUTA_SUC), limit=200_000)

print("Palabras en suc:", len(suc.key_to_index))
print("Dimensiones:", suc.vector_size)

Palabras en suc: 200000
Dimensiones: 100


In [45]:
#revisamos si suc distingue mayusculas y tildes

pruebas = ["españa", "España", "ESPAÑA", "espana", "méxico", "mexico", "Mexico", "también", "tambien", "gobierno", "Gobierno"]

revision_suc = pd.DataFrame({
    "palabra": pruebas,
    "esta_en_suc": [palabra in suc.key_to_index for palabra in pruebas],
    "posicion_en_vocabulario": [suc.key_to_index.get(palabra) for palabra in pruebas]
})

display(
    revision_suc.style
    .hide(axis="index")
    .set_caption("Revisión del vocabulario de SUC")
)

#cuantos tokens de cada corpus (ya normalizados) existen en suc
for nombre, oraciones in [("Noticias", oraciones_noticias), ("Quijote", oraciones_quijote)]:
    tokens = [token for oracion in oraciones for token in oracion]
    cobertura = np.mean([token in suc.key_to_index for token in tokens]) * 100
    print(f"{nombre}: {cobertura:.2f}% de los tokens estan en suc")

palabra,esta_en_suc,posicion_en_vocabulario
españa,True,541.000000
España,False,nan
ESPAÑA,False,nan
espana,True,67448.000000
méxico,True,797.000000
mexico,True,16941.000000
Mexico,False,nan
también,True,64.000000
tambien,True,4259.000000
gobierno,True,236.000000


Noticias: 98.79% de los tokens estan en suc
Quijote: 97.38% de los tokens estan en suc


## Decisiones de normalización

- Minúsculas: sí. SUC está completamente en minúsculas: "españa" está en el vocabulario (posición 541), pero "España" y "ESPAÑA" no. Si dejamos las mayúsculas, nuestras palabras no se podrían buscar en SUC y, además, "Gobierno" y "gobierno" serían dos tipos distintos con la mitad de contextos cada uno.
- Tildes: se conservan. SUC sí las distingue: "méxico" está en la posición 797 y "mexico" en la 16,941, es decir, la versión sin tilde es una variante poco frecuente (casi un error de escritura). Quitar tildes nos dejaría con formas poco comunes en SUC y, además, mezclaría palabras distintas ("él"/"el", "sí"/"si", "más"/"mas").
- Puntuación y números: se eliminan. Con la expresión regular solo quedan palabras (letras, tildes, ñ y ü). Los números aportan poco a los vecinos semánticos y llenarían el vocabulario de tipos muy raros.
- Compatibilidad con SUC: con esta normalización, el 98.79 % de los tokens de las noticias y el 97.38 % de los del Quijote existen en SUC, por lo que la comparación de las secciones 5 y 8 es justa.

## ¿Vale la pena quitar stopwords o lematizar antes de entrenar embeddings?

En este caso, no.

- Stopwords: Word2Vec cuenta posiciones, no palabras "importantes". Si quitamos "de", "la" o "que", palabras que estaban separadas por dos o tres posiciones quedan juntas y la ventana de contexto se deforma. Además, las stopwords dan información sintáctica (qué tipo de palabra suele ir antes o después) y Word2Vec ya trae el parámetro `sample` para submuestrear las palabras muy frecuentes sin destruir la oración.
- Lematización: en los laboratorios 1 y 2 servía para reducir el vocabulario en BoW y TF-IDF. Aquí perdería información porque si "gobiernos" y "gobernó" se colapsan en una sola forma, se pierde la diferencia entre sustantivo y verbo, entre plural y singular y entre pasado y presente, y el modelo ya no podría aprender que "gobernó" se parece a otros verbos en pasado. Y SUC no está lematizado, así que perderíamos compatibilidad.

**Tabla de estadísticas por corpus**

In [46]:
#documentos, oraciones, tokens y tipos de cada corpus

def estadisticas_corpus(nombre, n_documentos, oraciones):
    tokens = [token for oracion in oraciones for token in oracion]
    return {
        "corpus": nombre,
        "documentos": n_documentos,
        "oraciones": len(oraciones),
        "tokens": len(tokens),
        "tipos": len(set(tokens))
    }

tabla_corpus = pd.DataFrame([
    estadisticas_corpus("Noticias", len(corpus), oraciones_noticias),
    estadisticas_corpus("Don Quijote", len(documentos_quijote), oraciones_quijote)
])

display(
    tabla_corpus.style
    .hide(axis="index")
    .format({"documentos": "{:,}", "oraciones": "{:,}", "tokens": "{:,}", "tipos": "{:,}"})
    .set_caption("Tamaño de cada corpus")
)

corpus,documentos,oraciones,tokens,tipos
Noticias,"1,137","19,939","593,044","27,762"
Don Quijote,127,"9,399","380,962","22,882"


**Vocabulario con distintos valores de frecuencia mínima**

In [47]:
#vocabulario para min_count = 1, 2 y 5 en el corpus de noticias

frecuencias_noticias = Counter(token for oracion in oraciones_noticias for token in oracion)
total_tokens_noticias = sum(frecuencias_noticias.values())

resultados_min_count = []

for min_count in [1, 2, 5]:
    vocabulario = {palabra for palabra, f in frecuencias_noticias.items() if f >= min_count}
    tokens_fuera = sum(f for palabra, f in frecuencias_noticias.items() if f < min_count)

    resultados_min_count.append({
        "min_count": min_count,
        "tamano_vocabulario": len(vocabulario),
        "tokens_fuera": tokens_fuera,
        "tokens_fuera_%": tokens_fuera / total_tokens_noticias * 100
    })

tabla_min_count = pd.DataFrame(resultados_min_count)

display(
    tabla_min_count.style
    .hide(axis="index")
    .format({"tamano_vocabulario": "{:,}", "tokens_fuera": "{:,}", "tokens_fuera_%": "{:.2f}%"})
    .set_caption("Vocabulario del corpus de noticias según min_count")
)

min_count,tamano_vocabulario,tokens_fuera,tokens_fuera_%
1,"27,762",0,0.00%
2,"15,813","11,949",2.01%
5,"8,278","32,013",5.40%


Con min_count = 1, el vocabulario tiene 27,762 tipos, pero 11,949 de ellos (43%) aparecen una sola vez y solo representan el 2.01% de los tokens. Con min_count = 2, el vocabulario baja a 15,813 tipos, una reduccion del 43%, y con min_count = 5 baja a 8,278 tipos, una reduccion del 70%, perdiendo apenas el 5.40% de los tokens. Esto significa que el vocabulario es muy grande debido a la cantidad de palabras raras, siguiendo la distribucion de Zipf, mientras que el texto casi no se ve afectado. Una palabra que aparece una sola vez no tiene suficientes contextos para aprender un vector confiable. Por eso, min_count = 2 es un buen equilibrio, ya que conserva el 98% del texto y elimina casi la mitad de los tipos.

**Pares (palabra central, contexto) de skip-gram hechos a mano**

In [48]:
#generamos a mano los pares de skip-gram para una oracion de noticias

def generar_pares_skipgram(oracion, ventana):
    pares = []
    for i, central in enumerate(oracion):
        inicio = max(0, i - ventana)
        fin = min(len(oracion), i + ventana + 1)
        for j in range(inicio, fin):
            if j != i:
                pares.append((central, oracion[j]))
    return pares

#buscamos una oracion corta para que se pueda ver completa
oracion_ejemplo = next(oracion for oracion in oraciones_noticias if 7 <= len(oracion) <= 8)
pares_ejemplo = generar_pares_skipgram(oracion_ejemplo, ventana=2)

print("Oracion:", " ".join(oracion_ejemplo))
print("Tokens:", len(oracion_ejemplo), "| Pares con ventana 2:", len(pares_ejemplo))

#mostramos los pares agrupados por palabra central, como en la diapositiva de la clase
for central in oracion_ejemplo:
    contextos = [c for (w, c) in pares_ejemplo if w == central]
    print(f"({central}) -> {contextos}")

Oracion: viva aerobus es como un hijo para mí
Tokens: 8 | Pares con ventana 2: 26
(viva) -> ['aerobus', 'es']
(aerobus) -> ['viva', 'es', 'como']
(es) -> ['viva', 'aerobus', 'como', 'un']
(como) -> ['aerobus', 'es', 'un', 'hijo']
(un) -> ['es', 'como', 'hijo', 'para']
(hijo) -> ['como', 'un', 'para', 'mí']
(para) -> ['un', 'hijo', 'mí']
(mí) -> ['hijo', 'para']


**Cantidad de pares de entrenamiento del corpus completo**

In [49]:
#numero de pares para ventana 2 y ventana 5 sin generarlos todos
#para una oracion de n tokens, la palabra en la posicion i tiene min(i, w) vecinos a la izquierda y min(n-1-i, w) a la derecha

def contar_pares(oraciones, ventana):
    total = 0
    for oracion in oraciones:
        n = len(oracion)
        posiciones = np.arange(n)
        total += int((np.minimum(posiciones, ventana) + np.minimum(n - 1 - posiciones, ventana)).sum())
    return total

#verificamos la formula con el generador manual
assert contar_pares([oracion_ejemplo], 2) == len(pares_ejemplo)

tabla_pares = pd.DataFrame([
    {
        "ventana": ventana,
        "pares": contar_pares(oraciones_noticias, ventana),
        "tokens": total_tokens_noticias
    }
    for ventana in [2, 5]
])
tabla_pares["pares_por_token"] = tabla_pares["pares"] / tabla_pares["tokens"]

display(
    tabla_pares.style
    .hide(axis="index")
    .format({"pares": "{:,}", "tokens": "{:,}", "pares_por_token": "{:.2f}"})
    .set_caption("Pares de entrenamiento de skip-gram en el corpus de noticias")
)

ventana,pares,tokens,pares_por_token
2,"2,252,542","593,044",3.80
5,"5,333,292","593,044",8.99


Con ventana 2, el corpus de noticias produce 2,252,542 pares (3.80 por token), y con ventana 5 produce 5,333,292 pares (8.99 por token). Son menos de 4 y 10 pares por token porque las palabras que estan en los bordes de la oracion tienen menos vecinos. Con apenas 593,044 tokens se generan millones de ejemplos de entrenamiento, y estos crecen casi linealmente con el tamano de la ventana.

Word2Vec no necesita datos etiquetados porque la etiqueta se obtiene del mismo texto. La tarea consiste en predecir una palabra vecina a partir de la palabra central, y la respuesta correcta ya esta escrita en la oracion. Es un tipo de aprendizaje autosupervisado, por lo que no hace falta que una persona anote los datos. Cualquier texto sin etiquetar puede generar sus propios pares de entrenamiento (palabra central y contexto).

## 2. Co-ocurrencia, PPMI y SVD

**Matriz de co-ocurrencia palabra × palabra (ventana ±2, 2,000 palabras más frecuentes)**

In [50]:
#vocabulario de las 2,000 palabras mas frecuentes del corpus de noticias

N_PALABRAS = 2000
VENTANA_COOC = 2

palabras_cooc = [palabra for palabra, _ in frecuencias_noticias.most_common(N_PALABRAS)]
indice_cooc = {palabra: i for i, palabra in enumerate(palabras_cooc)}

#convertimos todo el corpus a indices (-1 para palabras fuera del top 2000)
#y guardamos a que oracion pertenece cada token para no cruzar los limites de la oracion
ids = np.array([indice_cooc.get(token, -1) for oracion in oraciones_noticias for token in oracion])
ids_oracion = np.repeat(np.arange(len(oraciones_noticias)), [len(oracion) for oracion in oraciones_noticias])

#matriz de co-ocurrencia: para cada distancia d sumamos los pares (i, i+d) de la misma oracion
matriz_cooc = np.zeros((N_PALABRAS, N_PALABRAS), dtype=np.float64)

for d in range(1, VENTANA_COOC + 1):
    mismo_contexto = (ids_oracion[:-d] == ids_oracion[d:]) & (ids[:-d] >= 0) & (ids[d:] >= 0)
    izquierda = ids[:-d][mismo_contexto]
    derecha = ids[d:][mismo_contexto]
    np.add.at(matriz_cooc, (izquierda, derecha), 1)
    np.add.at(matriz_cooc, (derecha, izquierda), 1)

total_celdas = matriz_cooc.size
celdas_cero = int((matriz_cooc == 0).sum())
dispersion_cooc = celdas_cero / total_celdas * 100

print("Forma de la matriz de co-ocurrencia:", matriz_cooc.shape)
print("Total de celdas:", f"{total_celdas:,}")
print("Celdas con valor cero:", f"{celdas_cero:,}")
print(f"Porcentaje de ceros: {dispersion_cooc:.2f}%")
print("Co-ocurrencias totales (cada par cuenta dos veces):", f"{int(matriz_cooc.sum()):,}")

Forma de la matriz de co-ocurrencia: (2000, 2000)
Total de celdas: 4,000,000
Celdas con valor cero: 3,753,448
Porcentaje de ceros: 93.84%
Co-ocurrencias totales (cada par cuenta dos veces): 1,534,452


In [51]:
#comparamos con la dispersion de la matriz BoW (documentos x palabras) como en el laboratorio 2
from sklearn.feature_extraction.text import CountVectorizer

#reconstruimos los documentos de noticias como listas de tokens
limites = np.cumsum([0] + oraciones_por_doc_noticias)
documentos_tokens = [
    [token for oracion in oraciones_noticias[limites[i]:limites[i + 1]] for token in oracion]
    for i in range(len(oraciones_por_doc_noticias))
]

vectorizador_bow = CountVectorizer(analyzer=lambda doc: doc)
matriz_bow = vectorizador_bow.fit_transform(documentos_tokens)

dispersion_bow = (1 - matriz_bow.nnz / (matriz_bow.shape[0] * matriz_bow.shape[1])) * 100

comparacion_dispersion = pd.DataFrame({
    "matriz": ["Co-ocurrencia (2,000 x 2,000)", "BoW de este notebook (sin lematizar)", "BoW del laboratorio 2 (lematizado)"],
    "forma": [str(matriz_cooc.shape), str(matriz_bow.shape), "(1137, 27799)"],
    "ceros_%": [dispersion_cooc, dispersion_bow, 99.30]
})

display(
    comparacion_dispersion.style
    .hide(axis="index")
    .format({"ceros_%": "{:.2f}%"})
    .set_caption("Dispersión de las matrices")
)

matriz,forma,ceros_%
"Co-ocurrencia (2,000 x 2,000)","(2000, 2000)",93.84%
BoW de este notebook (sin lematizar),"(1137, 27762)",99.14%
BoW del laboratorio 2 (lematizado),"(1137, 27799)",99.30%


La matriz de co-ocurrencia tiene una forma de 2,000 x 2,000 y un 93.84% de ceros, mientras que la matriz BoW tiene un 99.14% de ceros (99.30% en el laboratorio 2, con lematizacion). Ambas son dispersas, pero la matriz de co-ocurrencia es bastante mas densa. Esto se debe a que cada fila corresponde a una palabra frecuente que aparece en muchos contextos distintos, mientras que en BoW cada documento utiliza solo una pequena parte de las mas de 27 mil palabras del vocabulario. Ademas, la matriz de co-ocurrencia es cuadrada y su tamano no depende del numero de documentos.

**Similitud coseno entre filas de la matriz de co-ocurrencia**

In [52]:
#similitud coseno entre las filas de la matriz de conteos

def coseno(u, v):
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))

pares_cooc = [
    ("empresa", "compañía"),
    ("gobierno", "presidente"),
    ("banco", "crédito"),
    ("banco", "mujeres"),
    ("inflación", "tecnología")
]

resultados_cos = []
for a, b in pares_cooc:
    if a in indice_cooc and b in indice_cooc:
        resultados_cos.append({
            "palabra_1": a,
            "palabra_2": b,
            "coseno": coseno(matriz_cooc[indice_cooc[a]], matriz_cooc[indice_cooc[b]])
        })
    else:
        print(f"El par {a} / {b} no esta en el top {N_PALABRAS}")

display(
    pd.DataFrame(resultados_cos).style
    .hide(axis="index")
    .format({"coseno": "{:.4f}"})
    .set_caption("Similitud coseno con la matriz de conteos")
)

palabra_1,palabra_2,coseno
empresa,compañía,0.9621
gobierno,presidente,0.7647
banco,crédito,0.5753
banco,mujeres,0.4877
inflación,tecnología,0.9489


Solo coincide parcialmente con la intuicion. "empresa" y "compania" tienen un coseno de 0.96, y "gobierno" y "presidente" de 0.76, lo cual es razonable. Sin embargo, "inflacion" y "tecnologia", que no tienen relacion directa, obtuvieron 0.95, casi igual que el par de sinonimos. Por otro lado, "banco" y "credito", que si estan relacionadas, obtuvieron 0.58, apenas por encima de "banco" y "mujeres", con 0.49. Esto ocurre porque los conteos crudos estan dominados por las palabras mas frecuentes, como "de", "la" y "que". Dos palabras cualesquiera pueden compartir muchos de esos contextos, haciendo que sus vectores sean similares aunque no tengan el mismo significado.

**Palabras con más co-ocurrencias con "empresa" y "gobierno"**

In [53]:
#las 10 palabras que mas co-ocurren con cada palabra (conteos crudos)

def top_vecinos_matriz(matriz, palabra, k=10):
    fila = matriz[indice_cooc[palabra]].copy()
    fila[indice_cooc[palabra]] = -np.inf  #la palabra no cuenta como vecina de si misma
    posiciones = np.argsort(fila)[::-1][:k]
    return [(palabras_cooc[i], matriz[indice_cooc[palabra], i]) for i in posiciones]

tabla_top_conteos = pd.DataFrame({
    "empresa": [f"{p} ({int(v)})" for p, v in top_vecinos_matriz(matriz_cooc, "empresa")],
    "gobierno": [f"{p} ({int(v)})" for p, v in top_vecinos_matriz(matriz_cooc, "gobierno")]
})
tabla_top_conteos.index += 1

display(tabla_top_conteos.style.set_caption("Top 10 por co-ocurrencia cruda (conteo entre paréntesis)"))

,empresa,gobierno
1,la (192),el (214)
2,de (143),del (107)
3,una (62),de (84)
4,que (47),que (62)
5,en (42),y (56)
6,y (32),nacional (36)
7,a (28),en (30)
8,con (26),para (24)
9,para (18),al (24)
10,como (16),por (23)


Dominan las palabras funcionales, como articulos, preposiciones y conjunciones. Para "empresa" aparecen "la", "de", "una", "que", "en" e "y", mientras que para "gobierno" aparecen "el", "del", "de", "que" e "y". Solo "nacional" (36) aparece como una palabra con contenido significativo. Estas palabras co-ocurren con casi todo simplemente porque son las mas frecuentes del espanol. Por eso, los conteos crudos aportan poca informacion sobre el significado de "empresa" o "gobierno".

**PPMI implementado con NumPy**

In [54]:
#PPMI(w, c) = max(0, log2( P(w, c) / (P(w) * P(c)) ))

def calcular_ppmi(matriz):
    total = matriz.sum()
    p_wc = matriz / total
    p_w = matriz.sum(axis=1, keepdims=True) / total
    p_c = matriz.sum(axis=0, keepdims=True) / total

    with np.errstate(divide="ignore", invalid="ignore"):
        pmi = np.log2(p_wc / (p_w * p_c))

    #las celdas con conteo cero dan log(0) = -inf, y las dejamos en cero
    pmi[~np.isfinite(pmi)] = 0
    return np.maximum(0, pmi)

matriz_ppmi = calcular_ppmi(matriz_cooc)

print("Forma de la matriz PPMI:", matriz_ppmi.shape)
print(f"Porcentaje de ceros en PPMI: {(matriz_ppmi == 0).mean() * 100:.2f}%")
print(f"Valor maximo de PPMI: {matriz_ppmi.max():.2f}")

Forma de la matriz PPMI: (2000, 2000)
Porcentaje de ceros en PPMI: 95.05%
Valor maximo de PPMI: 12.28


In [55]:
#las 10 palabras mas asociadas segun PPMI

tabla_top_ppmi = pd.DataFrame({
    "empresa": [f"{p} ({v:.2f})" for p, v in top_vecinos_matriz(matriz_ppmi, "empresa")],
    "gobierno": [f"{p} ({v:.2f})" for p, v in top_vecinos_matriz(matriz_ppmi, "gobierno")]
})
tabla_top_ppmi.index += 1

display(tabla_top_ppmi.style.set_caption("Top 10 por PPMI (valor entre paréntesis)"))

#cosenos de los mismos pares pero con PPMI
resultados_cos_ppmi = []
for a, b in pares_cooc:
    if a in indice_cooc and b in indice_cooc:
        resultados_cos_ppmi.append({
            "palabra_1": a,
            "palabra_2": b,
            "coseno_conteos": coseno(matriz_cooc[indice_cooc[a]], matriz_cooc[indice_cooc[b]]),
            "coseno_ppmi": coseno(matriz_ppmi[indice_cooc[a]], matriz_ppmi[indice_cooc[b]])
        })

display(
    pd.DataFrame(resultados_cos_ppmi).style
    .hide(axis="index")
    .format({"coseno_conteos": "{:.4f}", "coseno_ppmi": "{:.4f}"})
    .set_caption("Coseno con conteos y con PPMI")
)

,empresa,gobierno
1,telecomunicaciones (5.45),corporativo (7.21)
2,llega (5.19),federal (5.55)
3,especializada (5.11),sánchez (5.34)
4,líder (4.97),nacional (5.26)
5,cualquier (4.71),iván (5.22)
6,estadounidense (4.69),funcionarios (5.14)
7,colombiana (4.58),petro (4.75)
8,criterios (4.56),mexicano (4.66)
9,española (4.47),estará (4.54)
10,somos (4.23),llamado (4.51)


palabra_1,palabra_2,coseno_conteos,coseno_ppmi
empresa,compañía,0.9621,0.2499
gobierno,presidente,0.7647,0.1765
banco,crédito,0.5753,0.0880
banco,mujeres,0.4877,0.1039
inflación,tecnología,0.9489,0.0866


La lista cambio por completo: desaparecieron las palabras funcionales y ahora aparecen palabras con contenido. Para "empresa" aparecen "telecomunicaciones", "lider", "estadounidense", "colombiana" y "espanola". Para "gobierno" aparecen "corporativo", "federal", "nacional", "petro", "funcionarios" y "mexicano". Esto ocurre porque PPMI no mide cuantas veces co-ocurren dos palabras, sino cuanto mas de lo esperado por azar se relacionan, al dividir entre P(w) y P(c). Una palabra como "de" aparece con casi todo, por lo que su asociacion no resulta sorprendente y su valor queda en cero.

Como contrapartida, PPMI favorece las palabras menos frecuentes, por lo que pueden aparecer algunas poco informativas, como "somos", "estara" y "llega", debido a que tienen pocos conteos. Tambien los cosenos bajaron mucho. Por ejemplo, "inflacion" y "tecnologia" pasaron de 0.95 a 0.09, mientras que "empresa" y "compania" pasaron de 0.96 a 0.25. El par de sinonimos sigue teniendo la similitud mas alta, pero "banco" y "credito", con 0.09, no se distinguen de un par sin relacion, que tiene 0.10. Esto se puede explicar por el tamano reducido del corpus.

**Reducción a 100 dimensiones con TruncatedSVD**

In [56]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=100, random_state=SEED)
vectores_svd = svd.fit_transform(matriz_ppmi)  #U * Sigma, una fila de 100 dimensiones por palabra

print("Forma de los vectores densos:", vectores_svd.shape)
print(f"Varianza explicada por las 100 dimensiones: {svd.explained_variance_ratio_.sum() * 100:.2f}%")

#guardamos los vectores como KeyedVectors para usarlos en la seccion 5
kv_svd = KeyedVectors(vector_size=100)
kv_svd.add_vectors(palabras_cooc, vectores_svd)
kv_svd.save("vectores_svd_ppmi.kv")

print("Vectores guardados en vectores_svd_ppmi.kv")
for palabra in ["empresa", "gobierno"]:
    vecinos = [p for p, _ in kv_svd.most_similar(palabra, topn=8)]
    print(f"Vecinos de {palabra} (SVD sobre PPMI):", vecinos)

Forma de los vectores densos: (2000, 100)
Varianza explicada por las 100 dimensiones: 24.74%
Vectores guardados en vectores_svd_ppmi.kv
Vecinos de empresa (SVD sobre PPMI): ['plataforma', 'compañía', 'tecnología', 'entidad', 'comunidad', 'alianza', 'firma', 'sociedad']
Vecinos de gobierno (SVD sobre PPMI): ['república', 'regulador', 'congreso', 'federal', 'banco', 'comisión', 'también', 'se']


Los 100 componentes explican solo el 24.74% de la varianza de la matriz PPMI, pero los vecinos ya tienen sentido. Para "empresa" aparecen "plataforma", "compania", "tecnologia", "entidad" y "firma". Para "gobierno" aparecen "republica", "regulador", "congreso", "federal" y "comision". 
Por que las filas de la matriz ya son vectores de palabras y que problemas resuelven PPMI y SVD? 

Cada fila describe una palabra segun los contextos en los que aparece. Esto se basa en la hipotesis distribucional, que establece que una palabra se conoce por las palabras que la rodean. Dos palabras con contextos similares tienen filas parecidas, por lo que el coseno entre sus filas permite medir su similitud. Sin embargo, estos vectores tienen dos problemas. Primero, los conteos crudos estan dominados por las palabras frecuentes, como "la" y "de". PPMI resuelve este problema al medir la asociacion entre palabras en relacion con lo que se esperaria por azar. Segundo, los vectores son muy grandes, con 2,000 dimensiones en este caso, y tienen aproximadamente un 94% de ceros y mucho ruido. SVD resuelve este problema al reducirlos a 100 dimensiones densas que conservan los principales patrones de variacion. De esta manera, palabras que nunca co-ocurren directamente, pero comparten contextos similares, pueden quedar cerca en el espacio vectorial. Esto permite que el modelo generalice y capture relaciones entre palabras.

## 3. Investigación

**CBOW vs. skip-gram**

CBOW predice la palabra central a partir del promedio de los vectores de su contexto, mientras que skip-gram hace lo contrario: a partir de la palabra central predice cada una de las palabras de su contexto. Se espera que **skip-gram funcione mejor con palabras poco frecuentes**, porque cada aparición de la palabra genera varios ejemplos de entrenamiento (uno por vecino) y la palabra rara no se diluye en un promedio, como sí pasa en CBOW. CBOW en cambio es más rápido y funciona bien con palabras frecuentes.

**Negative sampling**

Un softmax sobre todo el vocabulario obliga a calcular el denominador sumando sobre las |V| palabras (decenas o cientos de miles) en cada ejemplo de entrenamiento, lo cual es muy costoso. Negative sampling lo convierte en una clasificación binaria: para el par real (w, c) se quiere una probabilidad alta, y para solo k palabras "negativas" tomadas al azar (con probabilidad proporcional a la frecuencia^0.75) se quiere una probabilidad baja, por lo que en cada paso solo se actualizan k + 1 vectores de contexto.

La probabilidad de que c sea un contexto real de w es:

$$P(\text{real} \mid w, c) = \sigma(v_w \cdot c_c) = \frac{1}{1 + e^{-v_w \cdot c_c}}$$

donde $v_w$ es el vector de la palabra central (de la tabla de palabras) y $c_c$ es el vector de la palabra de contexto (de la tabla de contexto). Si el producto punto es grande, la probabilidad se acerca a 1 y el modelo considera que las palabras suelen aparecer juntas.

**Dos tablas de vectores**

Cada palabra cumple dos roles distintos: puede ser la palabra central o puede ser parte del contexto de otra. word2vec guarda un vector para cada rol (la matriz de palabras W y la matriz de contexto C). Si fuera una sola tabla, el producto $v_w \cdot v_w$ de una palabra consigo misma tendría que ser alto, a pesar de que una palabra casi nunca aparece en su propia ventana. Al final **se conserva la tabla de palabras (W)** y se descarta la de contexto (algunas implementaciones, como GloVe, promedian ambas).

**word2vec vs. GloVe vs. fastText**

* **word2vec** es predictivo y local: una red aprende vectores al predecir palabras vecinas dentro de una ventana, viendo el corpus ejemplo por ejemplo.
* **GloVe** usa conteos globales: construye la matriz de co-ocurrencia de todo el corpus y ajusta los vectores para que su producto punto se parezca al logaritmo del conteo ($v_i \cdot c_j + b_i + b_j \approx \log X_{ij}$).
* **fastText** extiende skip-gram con subpalabras: representa cada palabra como la suma de los vectores de sus n-gramas de caracteres (de 3 a 6 en SUC) más la palabra completa.

fastText es especialmente útil en español porque la morfología es muy rica (género, número y decenas de conjugaciones por verbo: "comer", "comía", "comeremos"). Muchas de esas formas son raras o nunca aparecen en el corpus, pero comparten subpalabras con formas conocidas, así que fastText les puede construir un vector (también a palabras con errores de dedo).

**Parámetros de `gensim.models.Word2Vec`**

* `vector_size`: dimensión de los vectores de palabra.
* `window`: distancia máxima entre la palabra central y las palabras de contexto que se consideran.
* `min_count`: se ignoran las palabras con frecuencia menor a este valor.
* `sg`: 1 para skip-gram y 0 para CBOW.
* `negative`: cantidad de palabras negativas por cada par positivo en negative sampling (0 lo desactiva).
* `sample`: umbral para el submuestreo de palabras muy frecuentes; las palabras por encima de ese umbral se descartan aleatoriamente durante el entrenamiento (por defecto 1e-3).
* `epochs`: cuántas veces se recorre todo el corpus durante el entrenamiento.

**Capas de embeddings de PyTorch**

* `nn.Embedding(num_embeddings, embedding_dim, padding_idx)`: es una tabla de búsqueda de tamaño `num_embeddings x embedding_dim` que recibe índices de palabras y devuelve las filas correspondientes. `padding_idx` indica el índice usado para el relleno (padding): su vector se inicializa en ceros y no recibe gradiente, por lo que no se modifica al entrenar.
* `nn.Embedding.from_pretrained(embeddings, freeze)`: crea la capa a partir de una matriz ya entrenada (por ejemplo SUC). Con `freeze=True` los vectores quedan congelados y no se actualizan, y con `freeze=False` se permite hacer fine-tuning.
* `nn.EmbeddingBag(mode)`: recibe una bolsa de índices por documento y devuelve un solo vector que combina los embeddings de la bolsa, con `mode="mean"`, `"sum"` o `"max"`, sin crear el tensor intermedio de cada palabra. Sirve para clasificar textos como el promedio de los embeddings de sus palabras.

**¿Por qué multiplicar un one-hot por E equivale a seleccionar una fila?** Si $x$ es un vector one-hot con un 1 en la posición $i$, entonces $xE = \sum_j x_j E_{j,:} = E_{i,:}$, porque todos los demás términos de la suma se multiplican por cero. Por eso un embedding es lo mismo que una capa lineal (sin sesgo) sobre un one-hot, pero se implementa como una búsqueda de la fila, que es mucho más barata que multiplicar una matriz enorme.

## 4. Entrenar Word2Vec y fastText

**Modelo base y variantes de Word2Vec sobre las noticias**

In [57]:
#configuracion base pedida: vector_size=100, window=5, min_count=2, sg=1, negative=5
#usamos workers=1 y una semilla fija para que el entrenamiento sea reproducible

EPOCAS = 15

CONFIG_BASE = dict(
    vector_size=100,
    window=5,
    min_count=2,
    sg=1,
    negative=5,
    epochs=EPOCAS,
    workers=1,
    seed=SEED
)

def entrenar_word2vec(oraciones, **cambios):
    config = {**CONFIG_BASE, **cambios}
    inicio = time.time()
    modelo = Word2Vec(sentences=oraciones, **config)
    return modelo, time.time() - inicio

#modelo base y 5 variantes, cambiando un solo parametro a la vez
variantes = {
    "base": {},
    "cbow (sg=0)": {"sg": 0},
    "ventana 2": {"window": 2},
    "ventana 10": {"window": 10},
    "dimension 50": {"vector_size": 50},
    "dimension 300": {"vector_size": 300}
}

modelos_w2v = {}
tiempos_w2v = {}

for nombre, cambios in variantes.items():
    modelos_w2v[nombre], tiempos_w2v[nombre] = entrenar_word2vec(oraciones_noticias, **cambios)
    print(f"{nombre}: entrenado en {tiempos_w2v[nombre]:.1f} segundos")

base: entrenado en 73.4 segundos
cbow (sg=0): entrenado en 21.1 segundos
ventana 2: entrenado en 33.7 segundos
ventana 10: entrenado en 115.1 segundos
dimension 50: entrenado en 67.1 segundos
dimension 300: entrenado en 111.4 segundos


**Palabras de control, configuraciones y vecinos**

In [58]:
#palabras de control relacionadas con las noticias (revisamos que esten en el vocabulario)

palabras_control = ["banco", "gobierno", "empresa", "mercado", "tecnología", "sostenibilidad"]

vocab_base = modelos_w2v["base"].wv.key_to_index
faltantes = [palabra for palabra in palabras_control if palabra not in vocab_base]
print("Palabras de control que faltan en el vocabulario:", faltantes)
print("Vocabulario del modelo base:", len(vocab_base))

Palabras de control que faltan en el vocabulario: []
Vocabulario del modelo base: 15813


In [59]:
#tabla con la configuracion y el tiempo de entrenamiento de cada variante

tabla_config = pd.DataFrame([
    {
        "variante": nombre,
        "sg": modelos_w2v[nombre].sg,
        "window": modelos_w2v[nombre].window,
        "vector_size": modelos_w2v[nombre].vector_size,
        "min_count": modelos_w2v[nombre].min_count,
        "negative": modelos_w2v[nombre].negative,
        "epochs": modelos_w2v[nombre].epochs,
        "vocabulario": len(modelos_w2v[nombre].wv),
        "tiempo_s": tiempos_w2v[nombre]
    }
    for nombre in variantes
])

display(
    tabla_config.style
    .hide(axis="index")
    .format({"tiempo_s": "{:.1f}", "vocabulario": "{:,}"})
    .set_caption("Configuración y tiempo de entrenamiento de cada variante")
)

variante,sg,window,vector_size,min_count,negative,epochs,vocabulario,tiempo_s
base,1,5,100,2,5,15,"15,813",73.4
cbow (sg=0),0,5,100,2,5,15,"15,813",21.1
ventana 2,1,2,100,2,5,15,"15,813",33.7
ventana 10,1,10,100,2,5,15,"15,813",115.1
dimension 50,1,5,50,2,5,15,"15,813",67.1
dimension 300,1,5,300,2,5,15,"15,813",111.4


In [60]:
#las 5 palabras mas cercanas a cada palabra de control en cada variante

vecinos_variantes = pd.DataFrame({
    palabra: {
        nombre: ", ".join(p for p, _ in modelos_w2v[nombre].wv.most_similar(palabra, topn=5))
        for nombre in variantes
    }
    for palabra in palabras_control
})

display(vecinos_variantes.style.set_caption("5 vecinos más cercanos por variante"))

,banco,gobierno,empresa,mercado,tecnología,sostenibilidad
base,"central, agrario, república, popular, sudameris","congreso, constitucional, gremial, ordena, electo","startup, veridas, emergente, conjuntamente, multinacional","sector, mercados, colombiano, moneda, accionario","computación, tecnológica, autentificación, aplicada, estandarización","descarbonización, estratégicas, creatividad, internacionalización, prioridad"
cbow (sg=0),"cordillera, dominicana, supervisor, grupo, serfinanza","congreso, gremial, partido, comité, consejo","plataforma, startup, compañía, iniciativa, convocatoria","país, territorio, sector, alcance, mundo","innovación, sociedad, tecnológica, digitalización, computación","innovación, digitalización, transformación, diversidad, sociedad"
ventana 2,"central, popular, gnb, sudameris, finandina","congreso, bce, regulador, federal, senado","startup, compañía, criptomoneda, veridas, participada","mercados, sector, conglomerado, ente, producto","computación, internet, utilizará, actualización, software","descarbonización, digitalización, globalidad, innovación, creatividad"
ventana 10,"central, agrario, finandina, invest, popular","joe, ordena, maduro, ratificado, fundamento","compañía, startup, seré, biotecnología, conjuntamente","colombiano, accionario, sumada, desconocido, laboral","revolucionaria, computación, estandarización, aplicada, neurálgico","ballabriga, medioambiental, prioritario, suscríbete, sociedad"
dimension 50,"central, agrario, sudameris, gnb, república","congreso, minem, constitucional, gremial, tono","compañía, startup, criptomoneda, plataforma, participada","sector, colombiano, creciente, desconocido, vacancia","estandarización, tecnológica, autentificación, ayudarle, computación","ciso, enfocados, prioridad, prioridades, remodelando"
dimension 300,"central, agrario, república, popular, sudameris","congreso, constitucional, biden, petro, gremial","compañía, startup, multinacional, criptomoneda, emergente","colombiano, mercados, accionario, sumada, desconocido","computación, esencia, tecnológica, autentificación, estandarización","estratégicas, prioridades, actuación, internacionalización, newsletter"


In [61]:
#cuantificamos "sintactico vs tematico": proporcion de vecinos que son otra forma de la misma palabra
#(comparten las primeras 4 letras con la palabra de control, por ejemplo banco / bancos / bancaria)
#usamos los 10 vecinos mas cercanos para tener mas datos

resultados_morfologia = []

for nombre in variantes:
    comparten = []
    for palabra in palabras_control:
        vecinos = [p for p, _ in modelos_w2v[nombre].wv.most_similar(palabra, topn=10)]
        comparten.extend(vecino[:4] == palabra[:4] for vecino in vecinos)

    resultados_morfologia.append({
        "variante": nombre,
        "vecinos_misma_raiz_%": np.mean(comparten) * 100
    })

display(
    pd.DataFrame(resultados_morfologia).style
    .hide(axis="index")
    .format({"vecinos_misma_raiz_%": "{:.1f}%"})
    .set_caption("Vecinos que comparten raíz con la palabra de control (top 10)")
)

variante,vecinos_misma_raiz_%
base,5.0%
cbow (sg=0),1.7%
ventana 2,1.7%
ventana 10,0.0%
dimension 50,3.3%
dimension 300,5.0%


**Dimensión y CBOW.** Cambiar la dimensión casi no cambia los vecinos: con 300 dimensiones "banco" tiene exactamente los mismos 5 vecinos que el modelo base, y con 50 son muy parecidos. En cambio, el entrenamiento sí cuesta más (127 s con 300 dimensiones contra 79 s del base). CBOW es unas 3 veces más rápido (24 s) pero con palabras poco frecuentes funciona peor: los vecinos de "banco" son "cordillera", "dominicana" y "serfinanza", mientras que skip-gram devuelve "central", "agrario" y "sudameris" (nombres de bancos). Por otro lado, para palabras abstractas como "sostenibilidad" CBOW da vecinos razonables (innovación, digitalización, transformación).

**¿Ventanas pequeñas dan vecinos más sintácticos y las grandes más temáticos?** Con la métrica de "vecinos con la misma raíz" no se ve una tendencia clara (ventana 2: 1.7%, base: 5.0%, ventana 10: 0.0%), porque en un corpus de solo 593 mil tokens los porcentajes son muy bajos y casi todos los vecinos son otras palabras. Pero al leer los vecinos sí se nota la diferencia entre funcional y temático:

* Con **ventana 2** los vecinos son palabras que podrían ocupar el mismo lugar en la oración, es decir, de la misma clase: "banco" → central, popular, gnb, sudameris, finandina (todos nombres de bancos); "gobierno" → congreso, bce, regulador, federal, senado (instituciones); "empresa" → startup, compañía, criptomoneda.
* Con **ventana 10** aparecen palabras que simplemente comparten la misma noticia, sin ser parecidas: "gobierno" → joe, ordena, maduro, ratificado; "sostenibilidad" → ballabriga, medioambiental, suscríbete; "mercado" → colombiano, sumada, desconocido.

Entonces la idea se cumple en el sentido de similitud funcional (ventana chica) contra asociación temática (ventana grande), aunque no en el sentido literal de "otras formas de la misma palabra".

**Word2Vec sobre el Quijote**

In [62]:
#entrenamos un modelo con la configuracion base sobre el Quijote

w2v_quijote, tiempo_quijote = entrenar_word2vec(oraciones_quijote)

print(f"Quijote: entrenado en {tiempo_quijote:.1f} segundos, vocabulario de {len(w2v_quijote.wv):,} palabras")

#5 palabras presentes en ambos corpus
palabras_comparar = ["señor", "dinero", "casa", "gobierno", "palabra"]

for palabra in palabras_comparar:
    en_noticias = palabra in modelos_w2v["base"].wv.key_to_index
    en_quijote = palabra in w2v_quijote.wv.key_to_index
    print(f"{palabra}: noticias={en_noticias}, quijote={en_quijote}")

Quijote: entrenado en 37.1 segundos, vocabulario de 11,711 palabras
señor: noticias=True, quijote=True
dinero: noticias=True, quijote=True
casa: noticias=True, quijote=True
gobierno: noticias=True, quijote=True
palabra: noticias=True, quijote=True


In [63]:
comparacion_corpus = pd.DataFrame({
    "palabra": palabras_comparar,
    "vecinos en noticias": [
        ", ".join(p for p, _ in modelos_w2v["base"].wv.most_similar(palabra, topn=6))
        for palabra in palabras_comparar
    ],
    "vecinos en el Quijote": [
        ", ".join(p for p, _ in w2v_quijote.wv.most_similar(palabra, topn=6))
        for palabra in palabras_comparar
    ]
})

display(
    comparacion_corpus.style
    .hide(axis="index")
    .set_caption("Vecinos más cercanos según el corpus de entrenamiento")
)

palabra,vecinos en noticias,vecinos en el Quijote
señor,"avicanna, adrian, brazo, dimayor, calderon, civic","jerónimo, mío, bagatele, manderecha, lléguese, suba"
dinero,"gastar, score, estorbo, guardar, cdt, hurto","dueño, traed, sabéis, pareciéndome, pagaba, capellán"
casa,"pasiva, teletrabajar, terraza, enfoca, notificación, filtro","aldea, corte, dejé, pueblo, id, hija"
gobierno,"congreso, constitucional, gremial, ordena, electo, tono","ínsula, condado, pago, barataria, regalo, pague"
palabra,"letras, fina, escaso, pagas, operamos, estricto","respondelle, hablase, desató, hablar, hablarle, suspirar"


Las mismas palabras tienen vecinos totalmente distintos según el corpus:

* **gobierno:** en las noticias → congreso, constitucional, gremial, ordena, electo; en el Quijote → ínsula, condado, barataria, pago, regalo. Aquí el "gobierno" es la ínsula Barataria que le prometen a Sancho, no una institución política.
* **casa:** en las noticias → pasiva, teletrabajar, terraza, notificación (casa como vivienda moderna); en el Quijote → aldea, corte, pueblo, hija.
* **señor:** en las noticias → avicanna, adrian, calderon, civic (aparece antes de nombres propios); en el Quijote → jerónimo, mío, manderecha, suba (forma de tratamiento dentro del diálogo).
* **palabra:** en las noticias → letras, fina, escaso, pagas; en el Quijote → respondelle, hablase, hablar, hablarle, suspirar (se asocia a hablar y responder).
* **dinero:** en las noticias → gastar, guardar, cdt, hurto; en el Quijote → dueño, pagaba, capellán (aunque esta lista es más ruidosa por ser un corpus pequeño).

Esto confirma la frase "el significado viene del corpus": un modelo de embeddings no aprende lo que una palabra "significa" en abstracto, sino cómo se usa en los textos con los que fue entrenado. Un mismo modelo entrenado con un español del siglo XVII y otro con noticias financieras modernas serían dos modelos distintos de la misma palabra. También hay que notar que con solo 380 mil tokens los vecinos de palabras poco frecuentes del Quijote son ruidosos ("pareciéndome", "traed").

**fastText de gensim sobre las noticias**

In [64]:
#entrenamos fastText con la misma configuracion base (n-gramas de 3 a 6 caracteres, como SUC)

inicio = time.time()
ft_noticias = FastText(
    sentences=oraciones_noticias,
    min_n=3,
    max_n=6,
    **CONFIG_BASE
)
tiempo_ft = time.time() - inicio

print(f"fastText: entrenado en {tiempo_ft:.1f} segundos, vocabulario de {len(ft_noticias.wv):,} palabras")

KeyboardInterrupt: 

In [ ]:
#palabras que no estan en el vocabulario: un error de dedo, una conjugacion poco comun y una palabra inventada

palabras_oov = ["gobirno", "privatizaríamos", "criptoempresa"]

for palabra in palabras_oov:
    print(f"\n{palabra}")
    print("  en vocabulario de word2vec:", palabra in modelos_w2v["base"].wv.key_to_index)
    print("  en vocabulario de fastText:", palabra in ft_noticias.wv.key_to_index)

    #fastText construye el vector a partir de los n-gramas de caracteres
    vector = ft_noticias.wv[palabra]
    print("  vector de fastText (primeras 5 dimensiones):", np.round(vector[:5], 3))
    print("  vecinos fastText:", [(p, round(s, 3)) for p, s in ft_noticias.wv.most_similar(palabra, topn=5)])

    #word2vec no tiene nada que consultar
    try:
        modelos_w2v["base"].wv[palabra]
    except KeyError as error:
        print("  word2vec:", error)


gobirno
  en vocabulario de word2vec: False
  en vocabulario de fastText: False
  vector de fastText (primeras 5 dimensiones): [ 0.322  0.108 -0.082  0.303  0.127]
  vecinos fastText: [('gobierno', 0.927), ('gobiernos', 0.76), ('cuaderno', 0.737), ('infierno', 0.698), ('gobernar', 0.687)]
  word2vec: "Key 'gobirno' not present"

privatizaríamos
  en vocabulario de word2vec: False
  en vocabulario de fastText: False
  vector de fastText (primeras 5 dimensiones): [-0.06   0.114  0.424 -0.011  0.345]
  vecinos fastText: [('otorgamos', 0.793), ('facilitamos', 0.788), ('garantizamos', 0.781), ('reforzamos', 0.774), ('imaginamos', 0.771)]
  word2vec: "Key 'privatizaríamos' not present"

criptoempresa
  en vocabulario de word2vec: False
  en vocabulario de fastText: False
  vector de fastText (primeras 5 dimensiones): [0.341 0.733 0.4   0.37  0.182]
  vecinos fastText: [('criptomoneda', 0.785), ('empresa', 0.782), ('cripto', 0.755), ('criptodivisas', 0.751), ('criptografía', 0.737)]
  word2v

Con **fastText** las tres palabras fuera de vocabulario obtienen un vector, porque el vector se construye sumando los de sus n-gramas de caracteres:

* **gobirno** (error de dedo) → gobierno (0.93), gobiernos, gobernar. Comparte casi todos sus n-gramas con "gobierno", así que lo recupera sin problema. También aparecen "cuaderno" e "infierno" por compartir la terminación.
* **privatizaríamos** (conjugación poco común) → otorgamos, facilitamos, garantizamos, reforzamos. Aquí fastText capturó la terminación de primera persona del plural (-amos) pero no la raíz "privatiz-", porque esa raíz es muy rara en el corpus. Es un ejemplo de que las subpalabras ayudan con la morfología pero no con el significado cuando faltan datos.
* **criptoempresa** (inventada) → criptomoneda, empresa, cripto, criptodivisas. Combinó muy bien los n-gramas de "cripto" y "empresa".

Con **Word2Vec** las tres palabras lanzan `KeyError`, porque trata cada palabra como un símbolo atómico con su propia fila en la tabla: si no se vio durante el entrenamiento, no existe un vector. Esa es la diferencia principal, y es la razón por la que fastText es útil en español, donde hay muchísimas formas conjugadas y derivadas que se ven poco. El costo es el tiempo de entrenamiento: 194 s contra 79 s del modelo base.

**Elección del mejor modelo propio**

In [ ]:
#guardamos el mejor modelo propio (vectores) para el resto del laboratorio

NOMBRE_MEJOR = "ventana 2"
w2v_mejor = modelos_w2v[NOMBRE_MEJOR]
kv_mejor = w2v_mejor.wv
kv_mejor.save("vectores_mejor_modelo.kv")

print("Mejor modelo:", NOMBRE_MEJOR)
print("Vectores guardados en vectores_mejor_modelo.kv")

Mejor modelo: ventana 2
Vectores guardados en vectores_mejor_modelo.kv


Elijo el modelo **Word2Vec skip-gram con ventana 2** (100 dimensiones, el resto igual al base) para el resto del laboratorio. En las 6 palabras de control fue el que dio los vecinos más limpios: casi todos son palabras del mismo tipo (bancos, instituciones, empresas o conceptos de sostenibilidad) y casi no tiene palabras sin relación como las que aparecen con ventana 10 ("joe", "suscríbete", "ballabriga") o con CBOW ("cordillera", "dominicana"). Al revisar los 30 vecinos a ojo, unos 25 me parecieron razonables, contra unos 22 del modelo base. Además entrena más rápido (45 s contra 79 s), y las ventanas chicas suelen funcionar mejor para analogías morfológicas. Las dimensiones 50 y 300 no mejoraron los vecinos, y 300 solo aumenta el costo y el riesgo de sobreajuste con un corpus de 593 mil tokens.

## 5. Similitud, vecinos y analogías

### 5.1 Similitud coseno y vecinos

In [ ]:
#los cuatro modelos que vamos a comparar

modelos = {
    "SVD sobre PPMI": kv_svd,
    "Word2Vec noticias": kv_mejor,
    "Word2Vec Quijote": w2v_quijote.wv,
    "fastText SUC": suc
}

tabla_modelos = pd.DataFrame({
    "modelo": list(modelos.keys()),
    "palabras": [len(kv.key_to_index) for kv in modelos.values()],
    "dimensiones": [kv.vector_size for kv in modelos.values()]
})

display(
    tabla_modelos.style
    .hide(axis="index")
    .format({"palabras": "{:,}"})
    .set_caption("Modelos que se comparan")
)

modelo,palabras,dimensiones
SVD sobre PPMI,"2,000",100
Word2Vec noticias,"15,813",100
Word2Vec Quijote,"11,711",100
fastText SUC,"200,000",100


In [ ]:
#funcion propia de similitud coseno (ya la habiamos definido en la seccion 2, la repetimos para tenerla a la mano)

def coseno(u, v):
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))

#verificamos que da lo mismo que similarity de gensim en 3 pares (con fastText SUC)
pares_verificacion = [("gobierno", "presidente"), ("banco", "dinero"), ("rey", "reina")]

verificacion = []
for a, b in pares_verificacion:
    propio = coseno(suc[a], suc[b])
    gensim_ = suc.similarity(a, b)
    verificacion.append({
        "par": f"{a} / {b}",
        "coseno_propio": propio,
        "similarity_gensim": gensim_,
        "coinciden": bool(np.isclose(propio, gensim_, atol=1e-6))
    })

display(
    pd.DataFrame(verificacion).style
    .hide(axis="index")
    .format({"coseno_propio": "{:.6f}", "similarity_gensim": "{:.6f}"})
    .set_caption("Verificación de la función coseno")
)

par,coseno_propio,similarity_gensim,coinciden
gobierno / presidente,0.677102,0.677102,True
banco / dinero,0.510769,0.510769,True
rey / reina,0.761168,0.761168,True


**Cinco vecinas más cercanas en los cuatro modelos**

In [ ]:
#escogemos 5 palabras que existan en los cuatro modelos
candidatas = ["gobierno", "dinero", "casa", "hombre", "tiempo", "país", "mujer", "banco", "guerra", "ciudad"]
palabras_vecinos = [p for p in candidatas if all(p in kv.key_to_index for kv in modelos.values())][:5]
print("Palabras elegidas:", palabras_vecinos)

tabla_vecinos = pd.DataFrame({
    nombre: {
        palabra: ", ".join(p for p, _ in kv.most_similar(palabra, topn=5))
        for palabra in palabras_vecinos
    }
    for nombre, kv in modelos.items()
})

display(tabla_vecinos.style.set_caption("5 vecinos más cercanos por modelo"))

Palabras elegidas: ['gobierno', 'dinero', 'casa', 'tiempo', 'banco']


,SVD sobre PPMI,Word2Vec noticias,Word2Vec Quijote,fastText SUC
gobierno,"república, regulador, congreso, federal, banco","congreso, bce, regulador, federal, senado","ínsula, condado, pago, barataria, regalo","elgobierno, gobier, go­bierno, ministro, gobiernos"
dinero,"punto, liquidez, cuenta, tiempo, momento","dispositivo, reembolso, coche, cajero, desea","dueño, traed, sabéis, pareciéndome, pagaba","dinerito, dineral, centavo, gastarlo, dinerillo"
casa,"coche, vivienda, página, fuerza, marca","pantalla, reformar, horario, pasiva, preparación","aldea, corte, dejé, pueblo, id","mansión, casita, cabaña, choza, habitación"
tiempo,"lo, tener, mismo, pero, primero","periodo, largos, período, incentivo, prolongado","deseado, anochecía, durante, gastado, día","mucho, rato, cuánto, tiempito, lapso"
banco,"bbva, grupo, bancolombia, occidente, central","central, popular, gnb, sudameris, finandina","acomodóse, encajó, mantón, túnica, cinta","hipotecario, bancos, bank, citibank, monetario"


La función `coseno` propia da exactamente el mismo valor que `similarity` de gensim en los 3 pares (por ejemplo, rey / reina = 0.761168).

Los vecinos reflejan el corpus y el método de cada modelo:

* **fastText SUC** da los vecinos más "correctos" y generales: dinero → dinerito, dineral, centavo; casa → mansión, casita, cabaña, choza; banco → hipotecario, bancos, citibank, monetario. Como usa subpalabras aparecen variantes ortográficas y diminutivos ("elgobierno", "dinerito", "tiempito"), y por haber visto 2.6 mil millones de palabras sus vecinos están bien ubicados en el campo semántico.
* **SVD sobre PPMI** y **Word2Vec de noticias** dan vecinos del dominio económico: banco → bbva, bancolombia, grupo (SVD) y central, popular, sudameris (Word2Vec), gobierno → congreso, regulador, federal. Pero con pocos datos también tienen ruido: para "tiempo" el SVD devuelve palabras funcionales (lo, tener, mismo, pero) y Word2Vec devuelve "dinero → dispositivo, reembolso, coche, cajero" y "casa → pantalla, reformar, horario".
* **Word2Vec Quijote** responde a la novela: gobierno → ínsula, barataria; banco → acomodóse, encajó, mantón (el banco donde uno se sienta, no la institución financiera).

**Pares de antónimos**

In [ ]:
#similitud de pares de antonimos en cada modelo

antonimos = [
    ("bueno", "malo"),
    ("subir", "bajar"),
    ("ganar", "perder"),
    ("público", "privado"),
    ("grande", "pequeño")
]

def similitud_o_nan(kv, a, b):
    if a in kv.key_to_index and b in kv.key_to_index:
        return coseno(kv[a], kv[b])
    return np.nan

tabla_antonimos = pd.DataFrame({
    nombre: [similitud_o_nan(kv, a, b) for a, b in antonimos]
    for nombre, kv in modelos.items()
}, index=[f"{a} / {b}" for a, b in antonimos])

display(
    tabla_antonimos.style
    .format("{:.3f}", na_rep="fuera de vocabulario")
    .set_caption("Similitud coseno de antónimos")
)

#como referencia, similitud promedio de pares de palabras al azar en fastText SUC
rng = np.random.default_rng(SEED)
palabras_suc = list(suc.key_to_index)[:20000]
azar = [coseno(suc[rng.choice(palabras_suc)], suc[rng.choice(palabras_suc)]) for _ in range(2000)]
print(f"Similitud promedio de pares al azar en SUC: {np.mean(azar):.3f}")

,SVD sobre PPMI,Word2Vec noticias,Word2Vec Quijote,fastText SUC
bueno / malo,fuera de vocabulario,0.660,0.668,0.717
subir / bajar,0.544,0.749,0.657,0.937
ganar / perder,fuera de vocabulario,0.675,0.497,0.689
público / privado,0.523,0.673,fuera de vocabulario,0.653
grande / pequeño,fuera de vocabulario,0.445,0.456,0.711


Similitud promedio de pares al azar en SUC: 0.207


Las similitudes de los antónimos son **altas**, entre 0.45 y 0.94 según el modelo, y en general más altas que el promedio de dos palabras al azar en SUC (0.207). En fastText SUC "subir" y "bajar" tienen 0.937, más que muchos pares de sinónimos, y "bueno" / "malo" tiene 0.717.

Esto pasa porque los embeddings miden similitud **distribucional**: dos palabras son parecidas si aparecen en contextos parecidos. "Subir" y "bajar" aparecen en las mismas frases ("los precios ___", "el dólar ___ ") y con las mismas palabras vecinas, así que son intercambiables en la oración. Lo que las separa (la polaridad) es un solo rasgo de significado, que el contexto casi no refleja. Por eso la similitud distribucional no es lo mismo que la sinonimia: mide que dos palabras se usan en los mismos lugares y sobre el mismo tema, no que signifiquen lo mismo (un sinónimo es intercambiable sin cambiar el sentido, un antónimo es intercambiable pero cambia el sentido al contrario).

### 5.2 Conjunto de similitud propio

In [ ]:
#20 pares de palabras con mi propio puntaje de similitud de 0 a 10
#(muy parecidos, relacionados y sin relacion)

pares_gold = [
    ("empresa", "compañía", 9.5),
    ("casa", "hogar", 9.0),
    ("guerra", "batalla", 8.0),
    ("dinero", "riqueza", 8.0),
    ("rey", "reina", 7.5),
    ("padre", "madre", 7.5),
    ("ciudad", "pueblo", 7.5),
    ("inflación", "precios", 7.0),
    ("gobierno", "presidente", 6.5),
    ("mercado", "precios", 6.5),
    ("economía", "mercado", 6.5),
    ("banco", "dinero", 6.0),
    ("hombre", "mujer", 6.0),
    ("subir", "bajar", 3.0),
    ("guerra", "paz", 3.0),
    ("bueno", "malo", 2.5),
    ("banco", "montaña", 0.5),
    ("dinero", "luna", 0.5),
    ("empresa", "caballo", 0.5),
    ("gobierno", "zapato", 0.0)
]

gold = pd.DataFrame(pares_gold, columns=["palabra_1", "palabra_2", "puntaje"])

for nombre, kv in modelos.items():
    gold[nombre] = [similitud_o_nan(kv, a, b) for a, b in zip(gold["palabra_1"], gold["palabra_2"])]

display(
    gold.style
    .hide(axis="index")
    .format({nombre: "{:.3f}" for nombre in modelos}, na_rep="OOV")
    .set_caption("Conjunto de similitud propio y coseno de cada modelo")
)

palabra_1,palabra_2,puntaje,SVD sobre PPMI,Word2Vec noticias,Word2Vec Quijote,fastText SUC
empresa,compañía,9.500000,0.693,0.674,0.479,0.824
casa,hogar,9.000000,0.356,0.411,OOV,0.608
guerra,batalla,8.000000,OOV,0.521,0.380,0.726
dinero,riqueza,8.000000,OOV,0.290,0.382,0.430
rey,reina,7.500000,OOV,0.779,0.359,0.761
padre,madre,7.500000,OOV,0.758,0.530,0.814
ciudad,pueblo,7.500000,OOV,0.390,0.360,0.572
inflación,precios,7.000000,0.531,0.487,OOV,0.690
gobierno,presidente,6.500000,0.416,0.404,OOV,0.677
mercado,precios,6.500000,0.218,0.297,OOV,0.658


In [ ]:
#correlacion de spearman entre mis puntajes y el coseno de cada modelo
from scipy.stats import spearmanr

#pares que existen en los cuatro modelos, para comparar todos sobre lo mismo
en_todos = gold[list(modelos)].notna().all(axis=1)

resultados_spearman = []
for nombre in modelos:
    disponibles = gold[nombre].notna()
    rho, p_valor = spearmanr(gold.loc[disponibles, "puntaje"], gold.loc[disponibles, nombre])

    fila = {
        "modelo": nombre,
        "pares_usados": int(disponibles.sum()),
        "pares_fuera_de_vocabulario": int((~disponibles).sum()),
        "spearman": rho,
        "p_valor": p_valor
    }

    if en_todos.sum() >= 5:
        fila[f"spearman_en_comun ({en_todos.sum()} pares)"] = spearmanr(
            gold.loc[en_todos, "puntaje"], gold.loc[en_todos, nombre]
        )[0]

    resultados_spearman.append(fila)

display(
    pd.DataFrame(resultados_spearman).style
    .hide(axis="index")
    .format({"spearman": "{:.3f}", "p_valor": "{:.4f}"}, na_rep="-")
    .set_caption("Correlación de Spearman con mi conjunto de similitud")
)

modelo,pares_usados,pares_fuera_de_vocabulario,spearman,p_valor
SVD sobre PPMI,8,12,0.293,0.4816
Word2Vec noticias,18,2,0.233,0.3530
Word2Vec Quijote,15,5,-0.146,0.6034
fastText SUC,20,0,0.564,0.0096


Los puntajes de similitud de los 20 pares los asigné yo y están en la tabla. **fastText SUC es el que más se parece a mi juicio**, con una correlación de Spearman de 0.564 (p = 0.0096) y todos los pares dentro del vocabulario. Los modelos propios están muy por debajo: Word2Vec de noticias 0.233 (18 pares, 2 fuera de vocabulario: "empresa / caballo" y "gobierno / zapato"), SVD sobre PPMI 0.293 (solo 8 pares, 12 fuera de vocabulario porque solo tiene 2,000 palabras) y Word2Vec del Quijote -0.146 (15 pares, 5 fuera de vocabulario). Ninguno de los tres modelos propios es estadísticamente significativo (p > 0.35), y los resultados del SVD y el Quijote se calcularon sobre menos pares, por lo que no son directamente comparables. El Quijote incluso tiene una correlación negativa porque "banco" y "montaña" salieron con 0.49 (el banco para sentarse), y las palabras económicas no existen en esa novela. Otro factor que baja las correlaciones de todos es que los modelos le dan similitud alta a los antónimos (bueno / malo, subir / bajar) mientras que yo les di puntajes bajos.

### 5.3 Analogías

In [ ]:
#funcion propia: "a es a b como c es a ?" usando el vector b - a + c con vectores normalizados

def analogia(kv, a, b, c, k=5, excluir=True):
    objetivo = kv.get_vector(b, norm=True) - kv.get_vector(a, norm=True) + kv.get_vector(c, norm=True)
    objetivo = objetivo / np.linalg.norm(objetivo)

    #coseno de todas las palabras del vocabulario contra el vector objetivo
    similitudes = kv.get_normed_vectors() @ objetivo
    resultado = []

    for i in np.argsort(-similitudes):
        palabra = kv.index_to_key[i]
        if excluir and palabra in (a, b, c):
            continue
        resultado.append((palabra, float(similitudes[i])))
        if len(resultado) == k:
            break

    return resultado

#verificamos contra most_similar de gensim en 3 analogias con fastText SUC
analogias_verificacion = [
    ("hombre", "rey", "mujer"),
    ("españa", "madrid", "francia"),
    ("comía", "comerá", "vivía")
]

for a, b, c in analogias_verificacion:
    propio = analogia(suc, a, b, c, k=5)
    de_gensim = suc.most_similar(positive=[b, c], negative=[a], topn=5)

    palabras_iguales = [p for p, _ in propio] == [p for p, _ in de_gensim]
    scores_iguales = np.allclose([s for _, s in propio], [s for _, s in de_gensim], atol=1e-5)

    print(f"{a} : {b} :: {c} : ?")
    print("  propio:", [(p, round(s, 4)) for p, s in propio])
    print("  gensim:", [(p, round(s, 4)) for p, s in de_gensim])
    print("  mismas palabras:", palabras_iguales, "| mismos scores:", scores_iguales)

hombre : rey :: mujer : ?
  propio: [('reina', 0.7628), ('princesa', 0.6654), ('consorte', 0.6648), ('sibila', 0.6532), ('isabel', 0.65)]
  gensim: [('reina', 0.7628), ('princesa', 0.6654), ('consorte', 0.6648), ('sibila', 0.6532), ('isabel', 0.65)]
  mismas palabras: True | mismos scores: True
españa : madrid :: francia : ?
  propio: [('toulouse', 0.8946), ('parís', 0.8866), ('lille', 0.8524), ('montpellier', 0.8369), ('rennes', 0.8341)]
  gensim: [('toulouse', 0.8946), ('parís', 0.8866), ('lille', 0.8524), ('montpellier', 0.8369), ('rennes', 0.8341)]
  mismas palabras: True | mismos scores: True
comía : comerá :: vivía : ?
  propio: [('vive', 0.8135), ('vivirá', 0.7986), ('regresará', 0.7438), ('mudará', 0.7385), ('habitará', 0.7334)]
  gensim: [('vive', 0.8135), ('vivirá', 0.7986), ('regresará', 0.7438), ('mudará', 0.7385), ('habitará', 0.7334)]
  mismas palabras: True | mismos scores: True


**Analogías propias**

In [ ]:
#24 analogias de 6 tipos: (tipo, a, b, c, respuesta esperada)

analogias = [
    ("género", "hombre", "rey", "mujer", "reina"),
    ("género", "padre", "madre", "hijo", "hija"),
    ("género", "actor", "actriz", "príncipe", "princesa"),
    ("género", "tío", "tía", "primo", "prima"),

    ("país-capital", "españa", "madrid", "francia", "parís"),
    ("país-capital", "italia", "roma", "alemania", "berlín"),
    ("país-capital", "portugal", "lisboa", "perú", "lima"),
    ("país-capital", "rusia", "moscú", "grecia", "atenas"),

    ("país-gentilicio", "españa", "español", "francia", "francés"),
    ("país-gentilicio", "italia", "italiano", "alemania", "alemán"),
    ("país-gentilicio", "méxico", "mexicano", "perú", "peruano"),
    ("país-gentilicio", "china", "chino", "japón", "japonés"),

    ("singular-plural", "casa", "casas", "país", "países"),
    ("singular-plural", "empresa", "empresas", "gobierno", "gobiernos"),
    ("singular-plural", "hombre", "hombres", "mujer", "mujeres"),
    ("singular-plural", "ciudad", "ciudades", "mes", "meses"),

    ("tiempos verbales", "comía", "comerá", "vivía", "vivirá"),
    ("tiempos verbales", "hablaba", "hablará", "jugaba", "jugará"),
    ("tiempos verbales", "trabajaba", "trabajará", "escribía", "escribirá"),
    ("tiempos verbales", "compró", "comprará", "vendió", "venderá"),

    ("adjetivos m-f", "bueno", "buena", "alto", "alta"),
    ("adjetivos m-f", "blanco", "blanca", "negro", "negra"),
    ("adjetivos m-f", "nuevo", "nueva", "viejo", "vieja"),
    ("adjetivos m-f", "español", "española", "francés", "francesa")
]

K_RANKING = 100  #buscamos la respuesta correcta dentro de las primeras 100 posiciones

registros = []
for nombre, kv in modelos.items():
    for tipo, a, b, c, esperada in analogias:
        if not all(p in kv.key_to_index for p in (a, b, c, esperada)):
            registros.append({"modelo": nombre, "tipo": tipo, "analogia": f"{a} : {b} :: {c} : ?", "esperada": esperada,
                              "top5": "fuera de vocabulario", "posicion": np.nan, "fuera_vocab": True})
            continue

        ranking = [p for p, _ in analogia(kv, a, b, c, k=K_RANKING)]
        posicion = ranking.index(esperada) + 1 if esperada in ranking else np.nan

        registros.append({"modelo": nombre, "tipo": tipo, "analogia": f"{a} : {b} :: {c} : ?", "esperada": esperada,
                          "top5": ", ".join(ranking[:5]), "posicion": posicion, "fuera_vocab": False})

resultados_analogias = pd.DataFrame(registros)

for nombre in modelos:
    detalle = resultados_analogias[resultados_analogias["modelo"] == nombre].drop(columns=["modelo", "fuera_vocab"])
    display(
        detalle.style
        .hide(axis="index")
        .format({"posicion": "{:.0f}"}, na_rep=f">{K_RANKING}")
        .set_caption(f"Analogías en {nombre} (posición de la respuesta correcta)")
    )

tipo,analogia,esperada,top5,posicion
género,hombre : rey :: mujer : ?,reina,fuera de vocabulario,>100
género,padre : madre :: hijo : ?,hija,fuera de vocabulario,>100
género,actor : actriz :: príncipe : ?,princesa,fuera de vocabulario,>100
género,tío : tía :: primo : ?,prima,fuera de vocabulario,>100
país-capital,españa : madrid :: francia : ?,parís,"italia, sede, unido, alemania, bilbao",14
país-capital,italia : roma :: alemania : ?,berlín,fuera de vocabulario,>100
país-capital,portugal : lisboa :: perú : ?,lima,fuera de vocabulario,>100
país-capital,rusia : moscú :: grecia : ?,atenas,fuera de vocabulario,>100
país-gentilicio,españa : español :: francia : ?,francés,fuera de vocabulario,>100
país-gentilicio,italia : italiano :: alemania : ?,alemán,fuera de vocabulario,>100


tipo,analogia,esperada,top5,posicion
género,hombre : rey :: mujer : ?,reina,"cárdenas, coordinador, lópez, gelsa, conversó",>100
género,padre : madre :: hijo : ?,hija,"robot, error, círculo, mensaje, alguien",>100
género,actor : actriz :: príncipe : ?,princesa,fuera de vocabulario,>100
género,tío : tía :: primo : ?,prima,fuera de vocabulario,>100
país-capital,españa : madrid :: francia : ?,parís,"lituania, lattes, murcia, león, aragón",>100
país-capital,italia : roma :: alemania : ?,berlín,"honduras, noruega, irlanda, lituania, barú",>100
país-capital,portugal : lisboa :: perú : ?,lima,fuera de vocabulario,>100
país-capital,rusia : moscú :: grecia : ?,atenas,fuera de vocabulario,>100
país-gentilicio,españa : español :: francia : ?,francés,"csi, stoxx, supremo, retrocede, ftse",>100
país-gentilicio,italia : italiano :: alemania : ?,alemán,"accionario, latino, blinov, persistirá, observadores",>100


tipo,analogia,esperada,top5,posicion
género,hombre : rey :: mujer : ?,reina,"antonomasia, patria, propio, marcela, infanta",100
género,padre : madre :: hijo : ?,hija,"ciencia, torrellas, ingratitud, bendita, pertinacia",>100
género,actor : actriz :: príncipe : ?,princesa,fuera de vocabulario,>100
género,tío : tía :: primo : ?,prima,fuera de vocabulario,>100
país-capital,españa : madrid :: francia : ?,parís,"doce, otubre, docena, llegamos, almodóvar",>100
país-capital,italia : roma :: alemania : ?,berlín,fuera de vocabulario,>100
país-capital,portugal : lisboa :: perú : ?,lima,fuera de vocabulario,>100
país-capital,rusia : moscú :: grecia : ?,atenas,fuera de vocabulario,>100
país-gentilicio,españa : español :: francia : ?,francés,"bartolomé, amador, pierres, vargas, martín",28
país-gentilicio,italia : italiano :: alemania : ?,alemán,"seguid, captivo, engáñaste, interrumpáis, entretanto",>100


tipo,analogia,esperada,top5,posicion
género,hombre : rey :: mujer : ?,reina,"reina, princesa, consorte, sibila, isabel",1
género,padre : madre :: hijo : ?,hija,"hija, esposa, hermana, nieta, cuñada",1
género,actor : actriz :: príncipe : ?,princesa,"princesa, emperatriz, reina, consorte, duquesa",1
género,tío : tía :: primo : ?,prima,"cuñada, hermana, sobrina, hermanastra, nieta",63
país-capital,españa : madrid :: francia : ?,parís,"toulouse, parís, lille, montpellier, rennes",2
país-capital,italia : roma :: alemania : ?,berlín,"berlín, munich, münster, múnich, núremberg",1
país-capital,portugal : lisboa :: perú : ?,lima,"lima, arequipa, iquitos, cusco, bogotá",1
país-capital,rusia : moscú :: grecia : ?,atenas,"salónica, atenas, tesalónica, patras, creta",2
país-gentilicio,españa : español :: francia : ?,francés,"francés, alemán, français, occitano, luxemburgués",1
país-gentilicio,italia : italiano :: alemania : ?,alemán,"alemán, austriaco, austríaco, checo, polaco",1


**Porcentaje de aciertos por tipo de analogía**

In [ ]:
#aciertos en top 1 y top 5 por tipo (las analogias fuera de vocabulario cuentan como fallo)

resultados_analogias["top1"] = resultados_analogias["posicion"] == 1
resultados_analogias["top5_ok"] = resultados_analogias["posicion"] <= 5

resumen_top1 = resultados_analogias.pivot_table(index="tipo", columns="modelo", values="top1", aggfunc="mean", sort=False) * 100
resumen_top5 = resultados_analogias.pivot_table(index="tipo", columns="modelo", values="top5_ok", aggfunc="mean", sort=False) * 100
cobertura = resultados_analogias.assign(en_vocab=~resultados_analogias["fuera_vocab"]).pivot_table(
    index="tipo", columns="modelo", values="en_vocab", aggfunc="mean", sort=False) * 100

#agregamos una fila con el total
for tabla, columna in [(resumen_top1, "top1"), (resumen_top5, "top5_ok")]:
    tabla.loc["TOTAL"] = resultados_analogias.groupby("modelo")[columna].mean() * 100
cobertura.loc["TOTAL"] = (~resultados_analogias["fuera_vocab"]).groupby(resultados_analogias["modelo"]).mean() * 100

orden = list(modelos)
display(resumen_top1[orden].style.format("{:.0f}%").set_caption("Aciertos en top 1"))
display(resumen_top5[orden].style.format("{:.0f}%").set_caption("Aciertos en top 5"))
display(cobertura[orden].style.format("{:.0f}%").set_caption("Analogías con todas sus palabras en el vocabulario"))

modelo,SVD sobre PPMI,Word2Vec noticias,Word2Vec Quijote,fastText SUC
tipo,,,,
género,0%,0%,0%,75%
país-capital,0%,0%,0%,50%
país-gentilicio,0%,0%,0%,100%
singular-plural,0%,0%,0%,50%
tiempos verbales,0%,0%,0%,75%
adjetivos m-f,0%,0%,0%,100%
TOTAL,0%,0%,0%,75%


modelo,SVD sobre PPMI,Word2Vec noticias,Word2Vec Quijote,fastText SUC
tipo,,,,
género,0%,0%,0%,75%
país-capital,0%,0%,0%,100%
país-gentilicio,0%,0%,0%,100%
singular-plural,25%,25%,0%,75%
tiempos verbales,0%,0%,0%,100%
adjetivos m-f,0%,0%,25%,100%
TOTAL,4%,4%,4%,92%


modelo,SVD sobre PPMI,Word2Vec noticias,Word2Vec Quijote,fastText SUC
tipo,,,,
género,0%,50%,50%,100%
país-capital,25%,50%,25%,100%
país-gentilicio,0%,100%,50%,100%
singular-plural,75%,100%,75%,100%
tiempos verbales,0%,0%,0%,100%
adjetivos m-f,0%,75%,75%,100%
TOTAL,17%,62%,46%,100%


Se usaron 24 analogías de 6 tipos (4 por tipo). **fastText SUC** acierta el 75% en top 1 y el 92% en top 5. Funcionó perfecto en país-gentilicio y adjetivos masculino-femenino (100% en top 1) y muy bien en tiempos verbales (75% en top 1, 100% en top 5). Falló más en:

* **país-capital** (50% en top 1): para "españa : madrid :: francia" el primero es "toulouse" y "parís" queda segundo. El vector incluye el contexto de Francia, así que aparecen ciudades francesas en general.
* **singular-plural** (50%): "casa : casas :: país" devolvió "comarcas" y "países" quedó en la posición 54.
* **género** (75%): "tío : tía :: primo" devolvió "cuñada" y "prima" quedó en la posición 63.

Los modelos propios casi no resuelven analogías: SVD sobre PPMI, Word2Vec de noticias y Word2Vec del Quijote tienen 0% en top 1 y solo 4% en top 5 (1 de 24). Además tienen poca cobertura (17%, 62% y 46% de las analogías con todas sus palabras en el vocabulario) y los tiempos verbales no se pueden evaluar en ninguno, porque formas como "vivirá" o "escribirá" casi no aparecen. Lo único que se logra es el patrón singular-plural ("empresa : empresas :: gobierno" → "gobiernos" en la posición 4 con SVD y 5 con Word2Vec de noticias), que es el patrón más regular y frecuente.

**Analogías sin excluir las palabras de la consulta**

In [ ]:
#repetimos 5 analogias sin excluir a, b y c del resultado

analogias_sin_excluir = [
    ("hombre", "rey", "mujer"),
    ("españa", "madrid", "francia"),
    ("casa", "casas", "país"),
    ("comía", "comerá", "vivía"),
    ("bueno", "buena", "alto")
]

registros_sin = []
for a, b, c in analogias_sin_excluir:
    for nombre in ["Word2Vec noticias", "fastText SUC"]:
        kv = modelos[nombre]
        if not all(p in kv.key_to_index for p in (a, b, c)):
            continue
        registros_sin.append({
            "analogia": f"{a} : {b} :: {c} : ?",
            "modelo": nombre,
            "top5 sin excluir": ", ".join(f"{p} ({s:.2f})" for p, s in analogia(kv, a, b, c, k=5, excluir=False)),
            "top5 excluyendo": ", ".join(p for p, _ in analogia(kv, a, b, c, k=5, excluir=True))
        })

display(
    pd.DataFrame(registros_sin).style
    .hide(axis="index")
    .set_caption("Analogías con y sin excluir las palabras de la consulta")
)

analogia,modelo,top5 sin excluir,top5 excluyendo
hombre : rey :: mujer : ?,Word2Vec noticias,"mujer (0.77), cárdenas (0.68), rey (0.66), coordinador (0.66), lópez (0.65)","cárdenas, coordinador, lópez, gelsa, conversó"
hombre : rey :: mujer : ?,fastText SUC,"reina (0.76), rey (0.68), princesa (0.67), consorte (0.66), sibila (0.65)","reina, princesa, consorte, sibila, isabel"
españa : madrid :: francia : ?,Word2Vec noticias,"madrid (0.73), francia (0.73), lituania (0.69), lattes (0.68), murcia (0.67)","lituania, lattes, murcia, león, aragón"
españa : madrid :: francia : ?,fastText SUC,"toulouse (0.89), parís (0.89), lille (0.85), montpellier (0.84), rennes (0.83)","toulouse, parís, lille, montpellier, rennes"
casa : casas :: país : ?,Word2Vec noticias,"país (0.63), casas (0.59), motocicletas (0.50), recogen (0.49), poblaciones (0.48)","motocicletas, recogen, poblaciones, cumplirse, micro"
casa : casas :: país : ?,fastText SUC,"país (0.76), comarcas (0.63), aledañas (0.63), fronterizas (0.62), rurales (0.61)","comarcas, aledañas, fronterizas, rurales, provincias"
comía : comerá :: vivía : ?,fastText SUC,"vive (0.81), vivirá (0.80), regresará (0.74), mudará (0.74), habitará (0.73)","vive, vivirá, regresará, mudará, habitará"
bueno : buena :: alto : ?,Word2Vec noticias,"alto (0.64), buena (0.58), amplia (0.45), excelente (0.45), mejora (0.43)","amplia, excelente, mejora, gran, empresarial"
bueno : buena :: alto : ?,fastText SUC,"alto (0.72), alta (0.64), elevada (0.56), sólida (0.54), escasa (0.54)","alta, elevada, sólida, escasa, mayor"


Si no se excluyen las palabras de la consulta, en el modelo de noticias el primer lugar siempre es una de ellas: "mujer" (0.77) para hombre : rey :: mujer, "madrid" y "francia" (0.73) para españa : madrid :: francia, "país" (0.63) para casa : casas :: país y "alto" (0.64) para bueno : buena :: alto. En fastText SUC pasa en 2 de las 5 ("país" 0.76 y "alto" 0.72), mientras que en las otras la respuesta correcta o plausible sí queda primera ("reina", "toulouse"/"parís").

Esto dice que el "≈" de rey − hombre + mujer ≈ reina **no es una igualdad**. El vector que resulta es b − a + c, y la diferencia (b − a) es pequeña comparada con los vectores originales, entonces el resultado queda muy cerca de c (y de b). La respuesta correcta está en esa zona, pero para encontrarla hay que descartar las palabras de la consulta. Así que la aritmética funciona de forma aproximada y depende de que la relación sea consistente y de que el modelo tenga suficientes datos, y no como una operación exacta.

## 6. Visualización del espacio

**Selección de palabras por grupo temático**

In [ ]:
#candidatas de 6 grupos tematicos; nos quedamos con las que existen en el mejor modelo propio y en fastText SUC

grupos_candidatos = {
    "países": ["españa", "méxico", "colombia", "argentina", "chile", "perú", "brasil", "francia", "alemania", "italia",
               "china", "japón", "rusia", "canadá", "venezuela", "ecuador", "bolivia", "uruguay", "portugal", "india",
               "panamá", "cuba", "inglaterra", "suiza", "corea", "paraguay", "guatemala", "bélgica", "noruega", "grecia", "turquía"],
    "meses": ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto", "septiembre", "octubre",
              "noviembre", "diciembre"],
    "números": ["uno", "dos", "tres", "cuatro", "cinco", "seis", "siete", "ocho", "nueve", "diez", "once", "doce",
                "trece", "catorce", "quince", "veinte", "treinta", "cuarenta", "cincuenta", "cien", "mil", "millón",
                "primero", "segundo", "tercero", "décimo", "sesenta", "ochenta", "noventa", "quinto", "sexto"],
    "verbos de movimiento": ["ir", "venir", "caminar", "correr", "salir", "entrar", "llegar", "volver", "subir",
                             "bajar", "viajar", "avanzar", "regresar", "mover", "partir", "cruzar", "saltar", "huir",
                             "pasar", "acercar", "alejar", "seguir", "trasladar", "desplazar", "llegó", "salió", "recorrer", "llevar", "traer", "marchar", "escapar"],
    "términos económicos": ["inflación", "banco", "mercado", "inversión", "crédito", "deuda", "impuestos", "precios",
                            "economía", "empresa", "comercio", "exportaciones", "bolsa", "tasa", "ingresos", "producción",
                            "crecimiento", "déficit", "capital", "divisas", "ahorro", "salarios", "consumo", "importaciones",
                            "finanzas", "presupuesto", "dólar", "ganancias", "pérdidas"],
    "profesiones": ["médico", "abogado", "ingeniero", "profesor", "maestro", "enfermero", "arquitecto", "periodista",
                    "director", "presidente", "ministro", "gerente", "empresario", "científico", "policía", "juez",
                    "músico", "pintor", "escritor", "soldado", "panadero", "carpintero", "cocinero", "pastor", "capitán",
                    "contador", "analista", "economista", "banquero", "político", "ejecutivo", "técnico", "consultor", "alcalde", "senador", "gobernador", "investigador", "diputado", "funcionario", "gerente"]
}

MAX_POR_GRUPO = 26

grupos = {}
for grupo, candidatas_grupo in grupos_candidatos.items():
    disponibles = [p for p in dict.fromkeys(candidatas_grupo) if p in kv_mejor.key_to_index and p in suc.key_to_index]
    grupos[grupo] = disponibles[:MAX_POR_GRUPO]

palabras_viz = [p for lista in grupos.values() for p in lista]
etiquetas_viz = [grupo for grupo, lista in grupos.items() for _ in lista]

tabla_grupos = pd.DataFrame({
    "grupo": list(grupos),
    "candidatas": [len(grupos_candidatos[g]) for g in grupos],
    "usadas": [len(grupos[g]) for g in grupos]
})

display(tabla_grupos.style.hide(axis="index").set_caption("Palabras por grupo (presentes en ambos modelos)"))
print("Total de palabras:", len(palabras_viz))

**Proyección con PCA y t-SNE**

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

modelos_viz = {
    "Word2Vec noticias (ventana 2)": kv_mejor,
    "fastText SUC": suc
}

colores = dict(zip(grupos, ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]))

#cuantas palabras rotulamos por grupo (el resto se queda sin etiqueta para que la grafica se lea)
N_ETIQUETAS = 5

proyecciones = {}

fig, ejes = plt.subplots(2, 2, figsize=(16, 13))

for fila, (nombre, kv) in enumerate(modelos_viz.items()):
    matriz = np.array([kv[p] for p in palabras_viz])

    proyecciones[(nombre, "PCA")] = PCA(n_components=2, random_state=SEED).fit_transform(matriz)
    proyecciones[(nombre, "t-SNE")] = TSNE(
        n_components=2, perplexity=20, init="pca", random_state=SEED, metric="cosine"
    ).fit_transform(matriz)

    for columna, metodo in enumerate(["PCA", "t-SNE"]):
        eje = ejes[fila, columna]
        puntos = proyecciones[(nombre, metodo)]

        for grupo, lista in grupos.items():
            posiciones = [i for i, e in enumerate(etiquetas_viz) if e == grupo]
            eje.scatter(puntos[posiciones, 0], puntos[posiciones, 1], s=30, color=colores[grupo], label=grupo, alpha=0.8)

            for i in posiciones[:N_ETIQUETAS]:
                eje.annotate(palabras_viz[i], puntos[i], fontsize=9, xytext=(3, 3), textcoords="offset points")

        eje.set_title(f"{nombre} - {metodo}")
        eje.set_xticks([])
        eje.set_yticks([])

ejes[0, 0].legend(loc="best", fontsize=9)
plt.tight_layout()

plt.show()

**¿En qué modelo se separan mejor los grupos?**

In [ ]:
#medimos la separacion de los grupos en el espacio original de 100 dimensiones (no en el 2D)
#silhouette: de -1 a 1, mas alto significa grupos mas compactos y mas separados entre si
#pureza de los 5 vecinos: de los 5 vecinos mas cercanos (entre las palabras elegidas) que proporcion es del mismo grupo

from sklearn.metrics import silhouette_score

resultados_separacion = []

for nombre, kv in modelos_viz.items():
    matriz = np.array([kv[p] for p in palabras_viz])
    normalizada = matriz / np.linalg.norm(matriz, axis=1, keepdims=True)

    similitudes = normalizada @ normalizada.T
    np.fill_diagonal(similitudes, -np.inf)
    vecinos = np.argsort(-similitudes, axis=1)[:, :5]
    etiquetas_array = np.array(etiquetas_viz)
    pureza = (etiquetas_array[vecinos] == etiquetas_array[:, None]).mean()

    fila = {
        "modelo": nombre,
        "silhouette (coseno)": silhouette_score(matriz, etiquetas_viz, metric="cosine"),
        "pureza de los 5 vecinos": pureza
    }

    #pureza por grupo
    for grupo in grupos:
        indices = np.where(etiquetas_array == grupo)[0]
        fila[f"pureza {grupo}"] = (etiquetas_array[vecinos[indices]] == grupo).mean()

    resultados_separacion.append(fila)

display(
    pd.DataFrame(resultados_separacion).style
    .hide(axis="index")
    .format({c: "{:.3f}" for c in pd.DataFrame(resultados_separacion).columns if c != "modelo"})
    .set_caption("Separación de los grupos temáticos")
)

Se usaron 142 palabras de 6 grupos (países, números, verbos de movimiento, términos económicos y profesiones con 26 cada uno, y los 12 meses), todas presentes en ambos modelos. **Los grupos esperados sí se forman en los dos modelos, pero se separan mucho mejor en fastText SUC**: el silhouette en el espacio original es 0.387 contra 0.183 del Word2Vec de noticias, y en promedio 96.6% de los 5 vecinos de cada palabra son de su mismo grupo contra 78.3%.

* En **Word2Vec de noticias** los meses (pureza 1.00), los países (0.93) y las profesiones (0.84) se agrupan bien, mientras que los números (0.79), los verbos de movimiento (0.77) y sobre todo los términos económicos (0.49) quedan más mezclados. Esto último tiene sentido: todo el corpus trata de economía, así que las palabras económicas aparecen en contextos muy distintos entre sí y quedan dispersas, y "banco" termina lejos del resto de su grupo.
* En **fastText SUC** todos los grupos están por encima de 0.92 de pureza. En PCA se ve que términos económicos y profesiones se solapan en una misma franja, pero en t-SNE quedan claramente separados.

**PCA vs. t-SNE.** PCA es una proyección lineal: conserva las direcciones de mayor varianza y por eso respeta la estructura global, pero con solo 2 componentes mucho de la información se pierde y los grupos se encimaban (por ejemplo, términos económicos y profesiones en SUC). t-SNE es no lineal y se enfoca en conservar los vecindarios: intenta que las palabras que son vecinas en 100 dimensiones también lo sean en 2D, y por eso separa mejor los grupos.

**¿Por qué se puede confiar en los grupos de t-SNE pero no en la distancia entre grupos?** Porque t-SNE solo optimiza que los vecinos cercanos se mantengan cerca. Para lo demás usa una distribución de cola pesada que separa los grupos y expande o comprime cada región a su conveniencia, así que el tamaño de un grupo, su densidad y la distancia entre grupos no significan nada y cambian con la semilla y la perplejidad. Se puede ver en las gráficas: en t-SNE de SUC los meses quedan cerca de los números, pero en PCA quedan cerca de los países; la posición relativa de los grupos es arbitraria, y lo que sí se puede leer es que las palabras de un mismo grupo están juntas.

## 8. Clasificación de noticias con embeddings